# 📊 Automated Portfolio Tracker

**Author:** Teckler Ndiany | **Date:** March 2026

**Live Dashboard:** [View Google Sheet](https://docs.google.com/spreadsheets/d/1sDtTYVAEHsS9N7Qc_DrYftNKr1XrlJRMezkLJp7ABQQ)

---

## Project Goal

Build an automated stock portfolio monitoring system that:
- Fetches real-time prices
- Calculates P&L and allocation
- Detects drift from target allocation
- Updates Google Sheets dashboard automatically

**Tech Stack:** Python | pandas | yfinance | gspread | Google Sheets API

## Phase 1: Data Fetcher
Fetch real-time stock prices using yfinance API

In [1]:
# Install required packages
!pip install yfinance pandas numpy

## Phase 2: Portfolio Analysis
Calculate position values, P&L, allocation, and drift detection

In [7]:
"""
Portfolio Tracker - Phase 1: Data Fetcher
"""

import yfinance as yf
import pandas as pd
from datetime import datetime
from typing import List

def fetch_stock_data(tickers: List[str], period: str = '1d') -> pd.DataFrame:
    """
    Fetch current stock prices for given tickers

    Args:
        tickers: List of stock ticker symbols
        period: Time period for data

    Returns:
        DataFrame with ticker, price, timestamp
    """

    if not tickers:
        raise ValueError("Tickers list cannot be empty")

    try:
        print(f" Fetching data for {len(tickers)} ticker(s)...")
        data = yf.download(tickers, period=period, progress=False)

        if data.empty:
            raise Exception("No data returned")

    except Exception as e:
        print(f" Error: {e}")
        return pd.DataFrame(columns=['ticker', 'price', 'timestamp'])

    # Handle single vs multiple tickers
    if len(tickers) == 1:
        latest_prices = {tickers[0]: data['Close'].iloc[-1]}
    else:
        latest_prices = data['Close'].iloc[-1].to_dict()

    # Build result
    result_data = []
    current_time = datetime.now()

    for ticker, price in latest_prices.items():
        result_data.append({
            'ticker': ticker,
            'price': round(float(price), 2),
            'timestamp': current_time
        })

    df = pd.DataFrame(result_data)
    print(f" Successfully fetched {len(df)} stocks")

    return df


# Test it!
print("="*60)
print("PORTFOLIO TRACKER - Data Fetcher Test")
print("="*60)

test_tickers = ['KO', 'NVDA', 'CAT']
print(f"\n Fetching: {test_tickers}")

df = fetch_stock_data(test_tickers)

if not df.empty:
    print("\n RESULTS:")
    print(df.to_string(index=False))

    print(f"\n{'Ticker':<10} {'Price':<15} {'Timestamp'}")
    print("-"*60)
    for _, row in df.iterrows():
        print(f"{row['ticker']:<10} ${row['price']:<14.2f} {row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print(" No data retrieved")

print("\n" + "="*60)

# Test error handling
print("\n Testing error handling...")
bad_df = fetch_stock_data(['INVALIDXYZ'])
print(f"Empty DataFrame: {bad_df.empty}")

PORTFOLIO TRACKER - Data Fetcher Test

 Fetching: ['KO', 'NVDA', 'CAT']
 Fetching data for 3 ticker(s)...


/tmp/ipykernel_496/2543815668.py:27: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, progress=False)


 Successfully fetched 3 stocks

 RESULTS:
ticker  price                  timestamp
   CAT 704.82 2026-03-10 07:07:52.054532
    KO  77.80 2026-03-10 07:07:52.054532
  NVDA 182.65 2026-03-10 07:07:52.054532

Ticker     Price           Timestamp
------------------------------------------------------------
CAT        $704.82         2026-03-10 07:07:52
KO         $77.80          2026-03-10 07:07:52
NVDA       $182.65         2026-03-10 07:07:52


 Testing error handling...
 Fetching data for 1 ticker(s)...


/tmp/ipykernel_496/2543815668.py:27: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, progress=False)
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALIDXYZ"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['INVALIDXYZ']: YFPricesMissingError('possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")')


 Error: No data returned
Empty DataFrame: True


## Phase 3: Google Sheets Integration
Automatically update dashboard with portfolio metrics

In [8]:
"""
Phase 2: Portfolio Analysis & Calculations
"""

import pandas as pd
from typing import Dict, List

class PortfolioAnalyzer:
    """Calculate portfolio metrics, P&L, and drift detection"""

    def __init__(self, holdings: List[Dict], target_allocation: Dict[str, float]):
        """
        Args:
            holdings: List of dicts with 'ticker', 'shares', 'cost_basis'
            target_allocation: Dict of ticker: target_weight% (e.g., {'KO': 20})
        """
        self.holdings = holdings
        self.target_allocation = target_allocation

    def calculate_position_value(self, shares: float, current_price: float) -> float:
        """Calculate current market value of a position"""
        return shares * current_price

    def calculate_pnl(self, current_value: float, cost_basis: float) -> Dict:
        """Calculate profit/loss in dollars and percentage"""
        pnl_dollars = current_value - cost_basis
        pnl_percent = (pnl_dollars / cost_basis) * 100 if cost_basis > 0 else 0

        return {
            'pnl_dollars': round(pnl_dollars, 2),
            'pnl_percent': round(pnl_percent, 2)
        }

    def calculate_allocation(self, position_value: float, total_value: float) -> float:
        """Calculate position weight as percentage of total portfolio"""
        return round((position_value / total_value) * 100, 2) if total_value > 0 else 0

    def check_drift(self, current_allocation: float, target_allocation: float,
                    threshold: float = 5.0) -> Dict:
        """
        Check if allocation has drifted beyond threshold

        Returns:
            Dict with 'needs_rebalancing', 'drift_amount', 'drift_percent'
        """
        drift = current_allocation - target_allocation
        drift_percent = abs(drift)
        needs_rebalancing = drift_percent > threshold

        return {
            'needs_rebalancing': needs_rebalancing,
            'drift_amount': round(drift, 2),
            'drift_percent': round(drift_percent, 2)
        }

    def analyze_portfolio(self, current_prices: pd.DataFrame) -> pd.DataFrame:
        """
        Complete portfolio analysis

        Args:
            current_prices: DataFrame from fetch_stock_data()

        Returns:
            DataFrame with full portfolio metrics
        """
        results = []

        # Create price lookup
        price_map = dict(zip(current_prices['ticker'], current_prices['price']))

        for holding in self.holdings:
            ticker = holding['ticker']
            shares = holding['shares']
            cost_basis_per_share = holding['cost_basis']

            # Get current price
            current_price = price_map.get(ticker, 0)

            # Calculate metrics
            total_cost = shares * cost_basis_per_share
            current_value = self.calculate_position_value(shares, current_price)
            pnl = self.calculate_pnl(current_value, total_cost)

            results.append({
                'ticker': ticker,
                'shares': shares,
                'cost_basis': cost_basis_per_share,
                'total_cost': round(total_cost, 2),
                'current_price': current_price,
                'current_value': round(current_value, 2),
                'pnl_dollars': pnl['pnl_dollars'],
                'pnl_percent': pnl['pnl_percent']
            })

        df = pd.DataFrame(results)

        # Calculate total portfolio value
        total_value = df['current_value'].sum()

        # Add allocation and drift
        df['current_allocation'] = df.apply(
            lambda row: self.calculate_allocation(row['current_value'], total_value),
            axis=1
        )

        df['target_allocation'] = df['ticker'].map(self.target_allocation)

        df['drift'] = df.apply(
            lambda row: self.check_drift(
                row['current_allocation'],
                row['target_allocation']
            ),
            axis=1
        )

        # Extract drift details
        df['drift_amount'] = df['drift'].apply(lambda x: x['drift_amount'])
        df['drift_percent'] = df['drift'].apply(lambda x: x['drift_percent'])
        df['needs_rebalancing'] = df['drift'].apply(lambda x: x['needs_rebalancing'])
        df.drop('drift', axis=1, inplace=True)

        return df


# TEST PHASE 2
print("="*80)
print("PHASE 2: PORTFOLIO ANALYSIS TEST")
print("="*80)

# Define your portfolio holdings
portfolio_holdings = [
    {'ticker': 'KO', 'shares': 100, 'cost_basis': 60.00},
    {'ticker': 'NVDA', 'shares': 30, 'cost_basis': 450.00},
    {'ticker': 'CAT', 'shares': 20, 'cost_basis': 340.00}
]

# Define target allocation (should sum to 100%)
target_allocation = {
    'KO': 33.33,
    'NVDA': 33.33,
    'CAT': 33.34
}

# Fetch current prices (reuse Phase 1 function)
tickers = [h['ticker'] for h in portfolio_holdings]
current_prices = fetch_stock_data(tickers)

# Analyze portfolio
analyzer = PortfolioAnalyzer(portfolio_holdings, target_allocation)
portfolio_df = analyzer.analyze_portfolio(current_prices)

# Display results
print("\n COMPLETE PORTFOLIO ANALYSIS")
print("="*80)
print(portfolio_df.to_string(index=False))

# Summary
total_cost = portfolio_df['total_cost'].sum()
total_value = portfolio_df['current_value'].sum()
total_pnl = total_value - total_cost
total_pnl_pct = (total_pnl / total_cost) * 100

print("\n" + "="*80)
print(" PORTFOLIO SUMMARY")
print("="*80)
print(f"Total Cost Basis:    ${total_cost:,.2f}")
print(f"Current Value:       ${total_value:,.2f}")
print(f"Total P&L:           ${total_pnl:,.2f} ({total_pnl_pct:+.2f}%)")

# Rebalancing alerts
print("\n REBALANCING ALERTS")
print("="*80)
needs_rebalancing = portfolio_df[portfolio_df['needs_rebalancing'] == True]

if not needs_rebalancing.empty:
    for _, row in needs_rebalancing.iterrows():
        direction = "OVER" if row['drift_amount'] > 0 else "UNDER"
        print(f"  {row['ticker']}: {direction}WEIGHT by {abs(row['drift_amount']):.2f}% "
              f"(Current: {row['current_allocation']:.2f}%, Target: {row['target_allocation']:.2f}%)")
else:
    print(" Portfolio is balanced - no rebalancing needed!")

print("\n" + "="*80)

PHASE 2: PORTFOLIO ANALYSIS TEST
 Fetching data for 3 ticker(s)...


/tmp/ipykernel_496/2543815668.py:27: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period=period, progress=False)


 Successfully fetched 3 stocks

 COMPLETE PORTFOLIO ANALYSIS
ticker  shares  cost_basis  total_cost  current_price  current_value  pnl_dollars  pnl_percent  current_allocation  target_allocation  drift_amount  drift_percent  needs_rebalancing
    KO     100        60.0      6000.0          77.80         7780.0       1780.0        29.67               28.44              33.33         -4.89           4.89              False
  NVDA      30       450.0     13500.0         182.65         5479.5      -8020.5       -59.41               20.03              33.33        -13.30          13.30               True
   CAT      20       340.0      6800.0         704.82        14096.4       7296.4       107.30               51.53              33.34         18.19          18.19               True

 PORTFOLIO SUMMARY
Total Cost Basis:    $26,300.00
Current Value:       $27,355.90
Total P&L:           $1,055.90 (+4.01%)

 REBALANCING ALERTS
  NVDA: UNDERWEIGHT by 13.30% (Current: 20.03%, Target: 33.33%)
  

In [5]:
"""
Phase 3: Google Sheets Integration
"""

# First, authenticate with Google
from google.colab import auth
from google.auth import default
import gspread

print("🔐 Authenticating with Google...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print("✅ Authenticated successfully!")

# Create or open a Google Sheet
sheet_name = "Portfolio Tracker Dashboard"

try:
    # Try to open existing sheet
    spreadsheet = gc.open(sheet_name)
    print(f"📄 Opened existing sheet: {sheet_name}")
except:
    # Create new sheet if doesn't exist
    spreadsheet = gc.create(sheet_name)
    print(f"📄 Created new sheet: {sheet_name}")

# Share with yourself (so you can view it)
print(f"\n🔗 Sheet URL: {spreadsheet.url}")

# Get or create worksheets
try:
    holdings_sheet = spreadsheet.worksheet("Current Holdings")
except:
    holdings_sheet = spreadsheet.add_worksheet("Current Holdings", rows=100, cols=20)

try:
    summary_sheet = spreadsheet.worksheet("Summary")
except:
    summary_sheet = spreadsheet.add_worksheet("Summary", rows=100, cols=20)

# Clear existing data
holdings_sheet.clear()
summary_sheet.clear()

print("\n📝 Writing data to Google Sheets...")

# Write portfolio holdings
holdings_data = [portfolio_df.columns.tolist()] + portfolio_df.values.tolist()
holdings_sheet.update('A1', holdings_data)

# Format headers
holdings_sheet.format('A1:M1', {
    "backgroundColor": {"red": 0.2, "green": 0.6, "blue": 0.9},
    "textFormat": {"bold": True, "foregroundColor": {"red": 1, "green": 1, "blue": 1}}
})

# Write summary
summary_data = [
    ["Portfolio Summary", ""],
    ["", ""],
    ["Metric", "Value"],
    ["Total Cost Basis", f"${total_cost:,.2f}"],
    ["Current Value", f"${total_value:,.2f}"],
    ["Total P&L ($)", f"${total_pnl:,.2f}"],
    ["Total P&L (%)", f"{total_pnl_pct:+.2f}%"],
    ["", ""],
    ["Rebalancing Alerts", ""],
]

# Add rebalancing alerts
if not needs_rebalancing.empty:
    for _, row in needs_rebalancing.iterrows():
        direction = "OVERWEIGHT" if row['drift_amount'] > 0 else "UNDERWEIGHT"
        summary_data.append([
            row['ticker'],
            f"{direction} by {abs(row['drift_amount']):.2f}%"
        ])
else:
    summary_data.append(["Status", "✅ Portfolio Balanced"])

summary_sheet.update('A1', summary_data)

# Format summary
summary_sheet.format('A1:B1', {
    "textFormat": {"bold": True, "fontSize": 14}
})
summary_sheet.format('A3:B3', {
    "backgroundColor": {"red": 0.2, "green": 0.6, "blue": 0.9},
    "textFormat": {"bold": True, "foregroundColor": {"red": 1, "green": 1, "blue": 1}}
})

print("\n✅ Data written successfully!")
print(f"\n🔗 View your dashboard: {spreadsheet.url}")
print("\n📌 Bookmark this URL - it auto-updates when you run this script!")

🔐 Authenticating with Google...
✅ Authenticated successfully!
📄 Created new sheet: Portfolio Tracker Dashboard

🔗 Sheet URL: https://docs.google.com/spreadsheets/d/1sDtTYVAEHsS9N7Qc_DrYftNKr1XrlJRMezkLJp7ABQQ

📝 Writing data to Google Sheets...


/tmp/ipykernel_496/937639866.py:51: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  holdings_sheet.update('A1', holdings_data)
/tmp/ipykernel_496/937639866.py:83: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  summary_sheet.update('A1', summary_data)



✅ Data written successfully!

🔗 View your dashboard: https://docs.google.com/spreadsheets/d/1sDtTYVAEHsS9N7Qc_DrYftNKr1XrlJRMezkLJp7ABQQ

📌 Bookmark this URL - it auto-updates when you run this script!


#  Automated Portfolio Tracker

**Live Dashboard:** [View Google Sheet](https://docs.google.com/spreadsheets/d/1sDtTYVAEHsS9N7Qc_DrYftNKr1XrlJRMezkLJp7ABQQ)

## Project Overview
Real-time stock portfolio monitoring system with automated Google Sheets dashboard.

### Features
-  Real-time price fetching via yfinance API
- P&L calculation ($ and %)
-  Allocation drift detection (5% threshold)
- Automated Google Sheets updates
- Rebalancing alerts

### Tech Stack
- Python (pandas, yfinance, gspread)
- Google Sheets API
- Google Colab

### Portfolio Tracked
- **KO** (Coca-Cola): 100 shares
- **NVDA** (Nvidia): 30 shares  
- **CAT** (Caterpillar): 20 shares